# Laboratorio de Compresion SVD

Notebook interactivo para explorar compresiones personalizadas de BERT.
Configura los parametros de compresion en la Seccion 1 y ejecuta el resto
para obtener un informe visual completo.

**Outputs por ejecucion:**
- Dashboard resumen (parametros, F1, top emociones afectadas)
- Analisis por emocion (waterfall + radar)
- Visualizacion de embeddings (t-SNE scatter + KDE)
- Comparacion de patrones de atencion
- Cirugia espectral (que valores singulares se cortaron)

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from sklearn.manifold import TSNE
from scipy.stats import gaussian_kde
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)

from src.data import load_goemotions
from src.data.dataset import NUM_LABELS, EMOTION_NAMES, MODEL_NAME
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    compute_singular_value_energy,
    count_parameters,
    get_compression_ratio,
    get_target_layer_names,
    filter_layer_names,
)

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 250,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})

# Paleta consistente
C_BASELINE = '#2ecc71'
C_COMPRESSED = '#e74c3c'
C_NEUTRAL = '#3498db'
C_BG = '#ecf0f1'
C_TEXT = '#2c3e50'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
results_dir = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(results_dir, exist_ok=True)
print(f'Device: {device}')

In [ ]:
# Cargar modelo, tokenizer, datos y espectro SVD
model_path = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-final')
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=NUM_LABELS, problem_type='multi_label_classification',
)
baseline_model.eval().to(device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

_, _, test_ds, _, data_collator = load_goemotions()
print(f'Test set: {len(test_ds)} ejemplos')

# Subsample para t-SNE
N_SAMPLES = 1500
rng = np.random.RandomState(42)
indices = rng.choice(len(test_ds), size=min(N_SAMPLES, len(test_ds)), replace=False)
indices.sort()
sub_test = test_ds.select(indices)
print(f'Subsample t-SNE: {len(sub_test)} ejemplos')

# Espectro SVD de todas las capas
all_target_names = get_target_layer_names(baseline_model)
energy_info = compute_singular_value_energy(baseline_model)
print(f'Capas analizadas: {len(energy_info)}')

# Evaluacion helper
eval_args = TrainingArguments(
    output_dir=os.path.join(results_dir, 'tmp_eval'),
    per_device_eval_batch_size=64, fp16=(device == 'cuda'), report_to='none',
)

def evaluate_model(model):
    trainer = Trainer(model=model, args=eval_args,
                      compute_metrics=compute_metrics, data_collator=data_collator)
    return trainer.evaluate(test_ds)

@torch.no_grad()
def extract_cls_embeddings(model, dataset, batch_size=64):
    model.eval()
    dev = next(model.parameters()).device
    all_embs, all_labels = [], []
    for i in range(0, len(dataset), batch_size):
        batch = data_collator([dataset[j] for j in range(i, min(i+batch_size, len(dataset)))])
        outputs = model(input_ids=batch['input_ids'].to(dev),
                        attention_mask=batch['attention_mask'].to(dev),
                        output_hidden_states=True)
        all_embs.append(outputs.hidden_states[-1][:, 0, :].cpu().numpy())
        all_labels.append(batch['labels'].numpy())
    return np.concatenate(all_embs), np.concatenate(all_labels)

print('Setup completo.')

---
## 1. Configuracion de compresion

Edita los parametros de la celda siguiente y ejecuta el resto del notebook.
Cada ejecucion genera un informe visual completo.

In [ ]:
# ================================================================
#              CONFIGURACION DE COMPRESION
# ================================================================
#
# COMPONENT - Que tipo de matrices comprimir:
#   'query'             Solo matrices Q de self-attention
#   'key'               Solo matrices K
#   'value'             Solo matrices V
#   'attention_output'  Solo proyeccion de salida de atencion
#   'intermediate'      Solo FFN intermedia (768 -> 3072)
#   'ffn_output'        Solo FFN salida (3072 -> 768)
#   'attention'         Todas las de atencion (Q+K+V+Out)
#   'ffn'               Todas las de FFN (Inter+Out)
#   None                TODAS las capas lineales del encoder
#
# LAYERS - Que capas del encoder comprimir:
#   None                Todas (0-11)
#   [0,1,2,3]           Solo capas tempranas
#   [4,5,6,7]           Solo capas intermedias
#   [8,9,10,11]         Solo capas tardias
#   [11]                Solo la ultima capa
#   list(range(6,12))   Capas 6 a 11
#
# RANK - Rango de truncamiento SVD (menor = mas compresion):
#   256, 128, 64, 32...
#
# ================================================================

COMPONENT = 'attention'       # <-- edita aqui
LAYERS = [8, 9, 10, 11]       # <-- edita aqui
RANK = 128                    # <-- edita aqui
CONFIG_NAME = None             # None = nombre auto-generado

# Texto de ejemplo para visualizacion de atencion
SAMPLE_TEXT = "I'm so happy and grateful for everything you've done, thank you!"

# ================================================================
# Nombre automatico si no se especifica
if CONFIG_NAME is None:
    comp_str = COMPONENT if COMPONENT else 'all'
    layer_str = f'L{LAYERS}' if LAYERS else 'L0-11'
    CONFIG_NAME = f'{comp_str} {layer_str} r={RANK}'

print(f'Configuracion: {CONFIG_NAME}')
print(f'  Componente: {COMPONENT or "todos"}')
print(f'  Capas: {LAYERS or "todas (0-11)"}')
print(f'  Rango: {RANK}')

In [ ]:
# Aplicar compresion y evaluar
target_names = filter_layer_names(all_target_names, component=COMPONENT, layers=LAYERS)

# Workaround: ffn_output matchea attention.output.dense
if COMPONENT == 'ffn_output':
    target_names = [n for n in target_names if 'attention' not in n]

print(f'Capas a comprimir: {len(target_names)}')
for n in target_names:
    print(f'  {n}')

print(f'\nComprimiendo...')
compressed_model = apply_svd_compression(baseline_model, rank=RANK, layer_names=target_names)
compressed_model.to(device)

params_orig = count_parameters(baseline_model)['total']
params_comp = count_parameters(compressed_model)['total']
params_removed = params_orig - params_comp
comp_ratio = get_compression_ratio(baseline_model, compressed_model)

print(f'Parametros: {params_orig:,} -> {params_comp:,} ({params_removed:,} eliminados)')
print(f'Ratio de compresion: {comp_ratio:.3f}x')

print(f'\nEvaluando baseline...')
metrics_bl = evaluate_model(baseline_model)
f1_bl = metrics_bl['eval_f1_macro']
f1_micro_bl = metrics_bl['eval_f1_micro']
per_emo_bl = np.array([metrics_bl[f'eval_f1_label_{i}'] for i in range(NUM_LABELS)])
print(f'  F1 macro: {f1_bl:.4f}')

print(f'Evaluando {CONFIG_NAME}...')
metrics_comp = evaluate_model(compressed_model)
f1_comp = metrics_comp['eval_f1_macro']
f1_micro_comp = metrics_comp['eval_f1_micro']
per_emo_comp = np.array([metrics_comp[f'eval_f1_label_{i}'] for i in range(NUM_LABELS)])
print(f'  F1 macro: {f1_comp:.4f}')
print(f'  Delta: {f1_comp - f1_bl:+.4f} ({(f1_comp/f1_bl - 1)*100:+.2f}%)')

---
## 2. Dashboard resumen

In [ ]:
fig = plt.figure(figsize=(20, 8))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1.2, 1.8], wspace=0.35)

# --- Panel 1: Donut de parametros ---
ax1 = fig.add_subplot(gs[0])
sizes = [params_comp, params_removed]
colors_donut = [C_BASELINE, '#e74c3c55']
wedges, texts = ax1.pie(
    sizes, colors=colors_donut, startangle=90,
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=3),
)
ax1.text(0, 0.08, f'{params_comp/1e6:.1f}M', ha='center', va='center',
         fontsize=20, fontweight='bold', color=C_TEXT)
ax1.text(0, -0.15, f'de {params_orig/1e6:.1f}M', ha='center', va='center',
         fontsize=11, color='gray')
ax1.set_title(f'Parametros retenidos\n({comp_ratio:.2f}x compresion)', pad=15)

# Leyenda donut
ax1.legend(
    [f'Retenidos ({params_comp/1e6:.1f}M)',
     f'Eliminados ({params_removed/1e6:.1f}M)'],
    loc='lower center', fontsize=9, frameon=False,
)

# --- Panel 2: Barras F1 macro/micro ---
ax2 = fig.add_subplot(gs[1])
x_pos = np.array([0, 1.2])
width = 0.45

bars_bl = ax2.bar(x_pos - width/2, [f1_bl, f1_micro_bl], width,
                  color=C_BASELINE, label='Baseline', edgecolor='white', linewidth=1.5)
bars_cp = ax2.bar(x_pos + width/2, [f1_comp, f1_micro_comp], width,
                  color=C_COMPRESSED, label=CONFIG_NAME, edgecolor='white', linewidth=1.5)

# Anotar valores
for bars in [bars_bl, bars_cp]:
    for bar in bars:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.4f}',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(['F1 Macro', 'F1 Micro'], fontsize=12)
ax2.set_ylim(0, max(f1_bl, f1_micro_bl) * 1.15)
ax2.set_title('Comparacion de rendimiento')
ax2.legend(fontsize=10)
ax2.set_ylabel('F1 Score')

# --- Panel 3: Top 10 emociones mas afectadas ---
ax3 = fig.add_subplot(gs[2])
deltas = per_emo_comp - per_emo_bl
sort_idx = np.argsort(deltas)  # mas negativo primero
top_n = min(12, NUM_LABELS)
top_idx = sort_idx[:top_n]

y_pos = np.arange(top_n)
bar_colors = [C_COMPRESSED if d < 0 else C_BASELINE for d in deltas[top_idx]]

ax3.barh(y_pos, deltas[top_idx], color=bar_colors, edgecolor='white',
         linewidth=1, height=0.7, alpha=0.85)
ax3.set_yticks(y_pos)
ax3.set_yticklabels([EMOTION_NAMES[i] for i in top_idx], fontsize=10)
ax3.axvline(x=0, color=C_TEXT, linewidth=1.5)
ax3.set_xlabel('Cambio en F1', fontsize=11)
ax3.set_title(f'Top {top_n} emociones mas afectadas')
ax3.invert_yaxis()

# Anotar valores
for i, idx in enumerate(top_idx):
    d = deltas[idx]
    ax3.text(d - 0.003 if d < 0 else d + 0.003, i,
             f'{d:+.3f}', va='center',
             ha='right' if d < 0 else 'left',
             fontsize=9, fontweight='bold', color=C_TEXT)

fig.suptitle(f'Dashboard de Compresion: {CONFIG_NAME}',
             fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'lab_dashboard.png'), bbox_inches='tight')
plt.show()

---
## 3. Analisis por emocion

In [ ]:
# Radar chart: F1 por emocion
fig, (ax_radar, ax_diff) = plt.subplots(1, 2, figsize=(20, 9),
                                         subplot_kw={'projection': 'polar'})

angles = np.linspace(0, 2 * np.pi, NUM_LABELS, endpoint=False).tolist()
angles += angles[:1]

for ax, title, vals_list, colors, labels in [
    (ax_radar, 'F1 absoluto por emocion',
     [per_emo_bl, per_emo_comp],
     [C_BASELINE, C_COMPRESSED],
     ['Baseline', CONFIG_NAME]),
]:
    for vals, color, label in zip(vals_list, colors, labels):
        v = vals.tolist() + [vals[0]]
        ax.plot(angles, v, 'o-', color=color, linewidth=2.5, markersize=3, label=label)
        ax.fill(angles, v, color=color, alpha=0.08)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(EMOTION_NAMES, fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_rticks([0.2, 0.4, 0.6, 0.8])
    ax.set_title(title, pad=25, fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=11)

# Radar de diferencia
diff_vals = (deltas).tolist() + [deltas[0]]
abs_diff = np.abs(deltas)
norm_diff = abs_diff / (abs_diff.max() + 1e-8)

# Colores por magnitud de cambio
for i in range(NUM_LABELS):
    a_start = angles[i]
    a_end = angles[i + 1]
    a_mid = (a_start + a_end) / 2
    color = C_COMPRESSED if deltas[i] < 0 else C_BASELINE
    ax_diff.bar(a_mid, abs_diff[i], width=(a_end - a_start) * 0.85,
                color=color, alpha=0.7, edgecolor='white', linewidth=0.5)

ax_diff.set_xticks(angles[:-1])
ax_diff.set_xticklabels(EMOTION_NAMES, fontsize=7)
ax_diff.set_title('Magnitud del cambio por emocion\n(rojo = degradacion, verde = mejora)',
                   pad=25, fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'lab_emotion_radar.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall / lollipop de cambio por emocion
sort_idx_full = np.argsort(deltas)

fig, ax = plt.subplots(figsize=(12, 10))

for i, idx in enumerate(sort_idx_full):
    bl_val = per_emo_bl[idx]
    cp_val = per_emo_comp[idx]
    delta = deltas[idx]

    # Linea conectora
    color = C_COMPRESSED if delta < 0 else C_BASELINE
    ax.plot([bl_val, cp_val], [i, i], color=color, linewidth=2.5,
            alpha=0.7, zorder=2)
    # Punto baseline
    ax.scatter(bl_val, i, color=C_BASELINE, s=70, zorder=3,
               edgecolors='white', linewidth=1.5)
    # Punto comprimido
    ax.scatter(cp_val, i, color=color, s=70, zorder=3,
               edgecolors='white', linewidth=1.5, marker='D')
    # Anotacion delta
    text_x = min(bl_val, cp_val) - 0.02
    ax.text(text_x, i, f'{delta:+.3f}', va='center', ha='right',
            fontsize=8, color=C_TEXT, fontweight='bold')

ax.set_yticks(range(NUM_LABELS))
ax.set_yticklabels([EMOTION_NAMES[i] for i in sort_idx_full], fontsize=10)
ax.set_xlabel('F1 Score', fontsize=13)
ax.set_title(f'Impacto por emocion: Baseline vs {CONFIG_NAME}\n'
             f'(ordenado por degradacion)', fontsize=15, fontweight='bold')
ax.invert_yaxis()
ax.set_xlim(-0.08, 1.0)

# Leyenda
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_BASELINE,
           markersize=12, label='Baseline'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor=C_COMPRESSED,
           markersize=10, label=CONFIG_NAME),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'lab_emotion_lollipop.png'), bbox_inches='tight')
plt.show()

---
## 4. Visualizacion de embeddings (t-SNE)

In [ ]:
# Extraer embeddings y proyectar con t-SNE conjunto
print('Extrayendo embeddings baseline...')
embs_bl, labels_bl = extract_cls_embeddings(baseline_model, sub_test)
print('Extrayendo embeddings comprimido...')
embs_comp, _ = extract_cls_embeddings(compressed_model, sub_test)

# t-SNE conjunto (coordenadas compartidas)
print('Ejecutando t-SNE conjunto...')
all_embs = np.concatenate([embs_bl, embs_comp], axis=0)
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000,
            random_state=42, init='pca', learning_rate='auto')
all_coords = tsne.fit_transform(all_embs)

n = len(sub_test)
coords_bl = all_coords[:n]
coords_comp = all_coords[n:]

# Emocion dominante + top 10
dominant_emotion = np.argmax(labels_bl, axis=1)
unique, counts = np.unique(dominant_emotion, return_counts=True)
top_indices = unique[np.argsort(-counts)][:10]
is_top = np.isin(dominant_emotion, top_indices)

print(f't-SNE completo: {all_coords.shape}')

In [ ]:
# Scatter plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

cmap = plt.cm.tab10
x_min, x_max = all_coords[:, 0].min() - 2, all_coords[:, 0].max() + 2
y_min, y_max = all_coords[:, 1].min() - 2, all_coords[:, 1].max() + 2

for ax, coords, title, f1 in [
    (axes[0], coords_bl, 'Baseline', f1_bl),
    (axes[1], coords_comp, CONFIG_NAME, f1_comp),
]:
    # Puntos grises (emociones no-top)
    other = ~is_top
    ax.scatter(coords[other, 0], coords[other, 1],
               c=[(0.85, 0.85, 0.85, 0.15)], s=3, rasterized=True)

    # Top emociones coloreadas
    for rank_i, emo_idx in enumerate(top_indices):
        mask = dominant_emotion == emo_idx
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=[cmap(rank_i)], s=12, alpha=0.6,
                   label=EMOTION_NAMES[emo_idx] if ax == axes[0] else None,
                   rasterized=True)

    ax.set_title(f'{title}\nF1 macro = {f1:.4f}', fontsize=14, fontweight='bold')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks([])
    ax.set_yticks([])

axes[0].legend(bbox_to_anchor=(-0.12, 0.5), loc='center right',
               fontsize=9, frameon=True, framealpha=0.9, markerscale=2.5)

fig.suptitle('Clusters de emociones: Baseline vs Comprimido\n'
             '(t-SNE conjunto, coordenadas compartidas)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'lab_tsne_scatter.png'), bbox_inches='tight')
plt.show()

In [ ]:
# KDE density comparison
top_for_kde = top_indices[:6]
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

for ax, coords, title, f1 in [
    (axes[0], coords_bl, 'Baseline', f1_bl),
    (axes[1], coords_comp, CONFIG_NAME, f1_comp),
]:
    ax.scatter(coords[:, 0], coords[:, 1],
               c=[(0.9, 0.9, 0.9, 0.1)], s=1, rasterized=True)

    for rank_i, emo_idx in enumerate(top_for_kde):
        mask = dominant_emotion == emo_idx
        if mask.sum() < 10:
            continue
        x, y = coords[mask, 0], coords[mask, 1]
        try:
            kde = gaussian_kde(np.vstack([x, y]), bw_method=0.3)
            xi = np.linspace(x_min, x_max, 100)
            yi = np.linspace(y_min, y_max, 100)
            xi, yi = np.meshgrid(xi, yi)
            zi = kde(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)
            color = cmap(rank_i)
            ax.contour(xi, yi, zi, levels=3, colors=[color],
                       linewidths=1.5, alpha=0.7)
            ax.contourf(xi, yi, zi, levels=3,
                        colors=[(*color[:3], 0.05), (*color[:3], 0.1),
                                (*color[:3], 0.15), (*color[:3], 0.2)])
        except np.linalg.LinAlgError:
            pass

    ax.set_title(f'{title}\nF1 macro = {f1:.4f}', fontsize=14, fontweight='bold')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks([])
    ax.set_yticks([])

# Leyenda KDE
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=cmap(i), alpha=0.5, label=EMOTION_NAMES[idx])
                  for i, idx in enumerate(top_for_kde)]
axes[0].legend(handles=legend_handles, bbox_to_anchor=(-0.12, 0.5),
               loc='center right', fontsize=9, frameon=True, framealpha=0.9)

fig.suptitle('Densidad de emociones: Baseline vs Comprimido\n'
             '(contornos KDE sobre t-SNE conjunto)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'lab_tsne_kde.png'), bbox_inches='tight')
plt.show()

---
## 5. Comparacion de patrones de atencion

In [ ]:
@torch.no_grad()
def get_attention_maps(model, text, layer_idx=11):
    """Extrae mapa de atencion (promedio de cabezas) para una capa."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64)
    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}
    outputs = model(**inputs, output_attentions=True)
    attn = outputs.attentions[layer_idx][0]  # (n_heads, seq, seq)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    return attn.cpu().numpy(), tokens

# Elegir capa de atencion a visualizar
# Si se comprimieron capas de atencion, usar la ultima comprimida
if LAYERS is not None:
    attn_layer = max(LAYERS)
else:
    attn_layer = 11

print(f'Texto: "{SAMPLE_TEXT}"')
print(f'Capa de atencion: {attn_layer}')

attn_bl, tokens = get_attention_maps(baseline_model, SAMPLE_TEXT, attn_layer)
attn_comp, _ = get_attention_maps(compressed_model, SAMPLE_TEXT, attn_layer)

# Promedio de cabezas
attn_bl_avg = attn_bl.mean(axis=0)
attn_comp_avg = attn_comp.mean(axis=0)
attn_diff = attn_comp_avg - attn_bl_avg

print(f'Tokens: {len(tokens)}')
print(f'Attention shape: {attn_bl.shape}')

In [ ]:
# Visualizacion de atencion: baseline, comprimido, diferencia
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for ax, data, title, cmap_name in [
    (axes[0], attn_bl_avg, 'Baseline', 'Blues'),
    (axes[1], attn_comp_avg, CONFIG_NAME, 'Oranges'),
    (axes[2], attn_diff, 'Diferencia', 'RdBu_r'),
]:
    if cmap_name == 'RdBu_r':
        vmax = np.abs(data).max()
        im = ax.imshow(data, cmap=cmap_name, vmin=-vmax, vmax=vmax, aspect='auto')
    else:
        im = ax.imshow(data, cmap=cmap_name, aspect='auto')

    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=7)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=7)
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.6)

fig.suptitle(f'Patrones de atencion (capa {attn_layer}, promedio de cabezas)\n'
             f'"{SAMPLE_TEXT[:60]}..."',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'lab_attention_comparison.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Atencion por cabeza: paneles individuales
n_heads = attn_bl.shape[0]
ncols = 4
nrows = (n_heads + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4.5))
if nrows == 1:
    axes = axes[np.newaxis, :]

for h in range(n_heads):
    row, col = divmod(h, ncols)
    ax = axes[row, col]

    diff_h = attn_comp[h] - attn_bl[h]
    vmax = np.abs(diff_h).max()
    im = ax.imshow(diff_h, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

    ax.set_title(f'Cabeza {h}', fontsize=11, fontweight='bold')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=5)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=5)
    plt.colorbar(im, ax=ax, shrink=0.7)

for idx in range(n_heads, nrows * ncols):
    row, col = divmod(idx, ncols)
    axes[row, col].set_visible(False)

fig.suptitle(f'Cambio en atencion por cabeza (capa {attn_layer})\n'
             f'Rojo = mas atencion, Azul = menos atencion tras compresion',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'lab_attention_per_head.png'), bbox_inches='tight')
plt.show()

---
## 6. Cirugia espectral: que valores singulares se cortaron

In [ ]:
# Agrupar capas comprimidas por tipo de componente
def get_component(name):
    if 'attention.self.query' in name: return 'Query (Q)'
    if 'attention.self.key' in name: return 'Key (K)'
    if 'attention.self.value' in name: return 'Value (V)'
    if 'attention.output.dense' in name: return 'Attn Output'
    if 'intermediate.dense' in name: return 'FFN Inter'
    if 'output.dense' in name and 'attention' not in name: return 'FFN Output'
    return 'Unknown'

comp_groups = {}
for name in target_names:
    comp = get_component(name)
    if comp not in comp_groups:
        comp_groups[comp] = []
    comp_groups[comp].append(name)

print(f'Componentes comprimidos: {list(comp_groups.keys())}')
for comp, names in comp_groups.items():
    print(f'  {comp}: {len(names)} capas')

In [ ]:
# Cirugia espectral: espectro antes/despues con zona de corte
n_comps = len(comp_groups)
ncols = min(n_comps, 3)
nrows = (n_comps + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 7, nrows * 5))
if nrows == 1 and ncols == 1:
    axes = np.array([[axes]])
elif nrows == 1:
    axes = axes[np.newaxis, :]
elif ncols == 1:
    axes = axes[:, np.newaxis]

comp_colors = {
    'Query (Q)': '#e74c3c', 'Key (K)': '#e67e22', 'Value (V)': '#f1c40f',
    'Attn Output': '#2ecc71', 'FFN Inter': '#3498db', 'FFN Output': '#9b59b6',
}

for idx, (comp, names) in enumerate(comp_groups.items()):
    row, col = divmod(idx, ncols)
    ax = axes[row, col]
    color = comp_colors.get(comp, '#3498db')

    # Promediar espectro sobre capas del mismo tipo
    all_sv = []
    all_cum = []
    for name in names:
        info = energy_info[name]
        all_sv.append(info['singular_values'].numpy())
        all_cum.append(info['cumulative_energy'].numpy())
    mean_sv = np.mean(all_sv, axis=0)
    mean_cum = np.mean(all_cum, axis=0)
    n_sv = len(mean_sv)

    # Barras: retenidas (color) + cortadas (gris)
    x = np.arange(n_sv)
    retained_mask = x < RANK
    ax.bar(x[retained_mask], mean_sv[retained_mask], color=color, alpha=0.8, width=1.0)
    ax.bar(x[~retained_mask], mean_sv[~retained_mask], color='#bdc3c7', alpha=0.5, width=1.0)

    # Linea de corte
    ax.axvline(x=RANK - 0.5, color='black', linewidth=2.5, linestyle='--', zorder=5)

    # Energia retenida
    energy_retained = mean_cum[min(RANK - 1, n_sv - 1)] * 100
    ax.text(RANK + 5, mean_sv.max() * 0.85,
            f'Energia retenida:\n{energy_retained:.1f}%',
            fontsize=11, fontweight='bold', color=C_TEXT,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9))
    ax.text(RANK - 2, mean_sv.max() * 0.7,
            f'rank={RANK}', fontsize=10, ha='right',
            fontweight='bold', color='black')

    ax.set_title(f'{comp} ({len(names)} capas)', fontweight='bold')
    ax.set_xlabel('Indice del valor singular')
    ax.set_ylabel('Valor singular')
    ax.set_xlim(-1, min(n_sv, RANK * 3))

# Ocultar paneles vacios
for idx in range(n_comps, nrows * ncols):
    row, col = divmod(idx, ncols)
    axes[row, col].set_visible(False)

fig.suptitle(f'Cirugia espectral: que valores singulares se cortaron\n'
             f'Color = retenido, gris = eliminado (rank = {RANK})',
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'lab_spectral_surgery.png'), bbox_inches='tight')
plt.show()

---
## 7. Mapa de capas comprimidas

In [ ]:
# Diagrama mostrando que capas fueron comprimidas
ALL_COMPS = ['Query (Q)', 'Key (K)', 'Value (V)', 'Attn Output', 'FFN Inter', 'FFN Output']
ALL_COLORS = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']

# Construir mapa 12x6 de capas comprimidas
compression_map = np.zeros((12, 6))  # 0 = no comprimida, 1 = comprimida
energy_map = np.zeros((12, 6))       # energia retenida

for name in target_names:
    layer_idx = int(name.split('.')[3])
    comp = get_component(name)
    if comp in ALL_COMPS:
        comp_idx = ALL_COMPS.index(comp)
        compression_map[layer_idx, comp_idx] = 1
        cum_e = energy_info[name]['cumulative_energy'].numpy()
        energy_map[layer_idx, comp_idx] = cum_e[min(RANK - 1, len(cum_e) - 1)] * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Panel 1: Mapa binario de compresion
cmap_binary = plt.cm.colors.ListedColormap([C_BG, '#e74c3c88'])
sns.heatmap(compression_map, annot=False, cmap=cmap_binary,
            xticklabels=ALL_COMPS, yticklabels=[f'Capa {i}' for i in range(12)],
            linewidths=2, linecolor='white', cbar=False, ax=ax1)

# Marcar con check las comprimidas
for i in range(12):
    for j in range(6):
        if compression_map[i, j] == 1:
            ax1.text(j + 0.5, i + 0.5, 'SVD', ha='center', va='center',
                     fontsize=9, fontweight='bold', color='#c0392b')
        else:
            ax1.text(j + 0.5, i + 0.5, '-', ha='center', va='center',
                     fontsize=10, color='#bdc3c7')

ax1.set_title(f'Capas comprimidas\n({int(compression_map.sum())}/{12*6} capas)',
              fontweight='bold')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=35, ha='right')

# Panel 2: Energia retenida en capas comprimidas
# Mascara: solo mostrar capas comprimidas
masked_energy = np.where(compression_map == 1, energy_map, np.nan)

sns.heatmap(masked_energy, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=80, vmax=100,
            xticklabels=ALL_COMPS, yticklabels=[f'Capa {i}' for i in range(12)],
            linewidths=2, linecolor='white',
            cbar_kws={'label': 'Energia retenida (%)', 'shrink': 0.8},
            ax=ax2, mask=compression_map == 0)
# Celdas no comprimidas en gris
for i in range(12):
    for j in range(6):
        if compression_map[i, j] == 0:
            ax2.add_patch(plt.Rectangle((j, i), 1, 1, fill=True,
                                         facecolor=C_BG, edgecolor='white', linewidth=2))

ax2.set_title(f'Energia retenida (rank={RANK})\n'
              f'(gris = no comprimida)', fontweight='bold')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=35, ha='right')

fig.suptitle(f'Mapa de compresion: {CONFIG_NAME}', fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'lab_compression_map.png'), bbox_inches='tight')
plt.show()

---
## 8. Exportar resultados

In [ ]:
# Exportar resultados
result_dict = {
    'config_name': CONFIG_NAME,
    'component': COMPONENT or 'all',
    'layers': str(LAYERS) if LAYERS else 'all',
    'rank': RANK,
    'params_original': params_orig,
    'params_compressed': params_comp,
    'compression_ratio': comp_ratio,
    'f1_macro_baseline': f1_bl,
    'f1_macro_compressed': f1_comp,
    'f1_delta': f1_comp - f1_bl,
    'f1_retention_pct': f1_comp / f1_bl * 100,
    'n_layers_compressed': len(target_names),
}
df_result = pd.DataFrame([result_dict])

# Per-emotion results
emo_data = []
for i, emo in enumerate(EMOTION_NAMES):
    emo_data.append({
        'emotion': emo,
        'f1_baseline': per_emo_bl[i],
        'f1_compressed': per_emo_comp[i],
        'delta': deltas[i],
    })
df_emo = pd.DataFrame(emo_data).sort_values('delta')

csv_path = os.path.join(results_dir, 'lab_results.csv')
df_result.to_csv(csv_path, index=False)
emo_csv_path = os.path.join(results_dir, 'lab_per_emotion.csv')
df_emo.to_csv(emo_csv_path, index=False)

print(f'Guardado: {csv_path}')
print(f'Guardado: {emo_csv_path}')

In [ ]:
# Resumen final
figures = [
    'lab_dashboard.png',
    'lab_emotion_radar.png',
    'lab_emotion_lollipop.png',
    'lab_tsne_scatter.png',
    'lab_tsne_kde.png',
    'lab_attention_comparison.png',
    'lab_attention_per_head.png',
    'lab_spectral_surgery.png',
    'lab_compression_map.png',
]

print('=' * 65)
print(f'RESUMEN — Laboratorio de Compresion')
print('=' * 65)
print(f'\nConfiguracion: {CONFIG_NAME}')
print(f'  Componente: {COMPONENT or "todos"}')
print(f'  Capas: {LAYERS or "todas"}')
print(f'  Rango: {RANK}')
print(f'  Capas comprimidas: {len(target_names)}')
print(f'\nResultados:')
print(f'  Parametros: {params_orig:,} -> {params_comp:,} ({comp_ratio:.3f}x)')
print(f'  F1 macro:   {f1_bl:.4f} -> {f1_comp:.4f} ({f1_comp-f1_bl:+.4f})')
print(f'  Retencion:  {f1_comp/f1_bl*100:.2f}%')
print(f'\nEmociones mas afectadas:')
for _, row in df_emo.head(5).iterrows():
    print(f'  {row["emotion"]:<15s}: {row["delta"]:+.4f}')
print(f'\nFiguras generadas ({len(figures)}):')
for fig_name in figures:
    path = os.path.join(results_dir, fig_name)
    status = 'ok' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}] {fig_name}')

# Limpiar modelo comprimido
del compressed_model
if device == 'cuda':
    torch.cuda.empty_cache()
print(f'\nMemoria liberada.')